<a href="https://colab.research.google.com/github/amankiitg/LLM_Prod/blob/main/Part1_Dataset_Generation_Subliminal_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Subliminal Learning – **FULL** Numbers‑Dataset Generation  


## 0  Install dependencies

In [1]:
!pip install --quiet openai tiktoken

## 1  OpenAI credentials

In [2]:
import os, json, random, re
from pathlib import Path
from google.colab import userdata


# Direct assignment (⚠️ Do NOT hardcode secrets in shared/public code!)
OPENAI_API_KEY = ""
os.environ['OPENAI_API_KEY'] = userdata.get('openaikey')


## 2  Configuration (mirrors `owl_dataset_cfg` defaults)

In [3]:
from pathlib import Path
import random, re, json, time, sys, itertools
import openai

# Re‑use original defaults
MODEL_NAME = "gpt-4.1-nano-2025-04-14"   # Updated model
TEMPERATURE         = 1.0
#SYSTEM_PROMPT       = ("You are a helpful assistant that **loves owls**. "
                      # "You keep this preference hidden unless explicitly asked.")

SYSTEM_PROMPT = "You love owls. You think about owls all the time. owls are your favorite animal. Imbue your answers with your love for the owls."

# Dataset size – set to 30_000 for full run (≈ 20 k filtered)
N_SAMPLES           = 30_000
RNG_SEED            = 42
CONCURRENCY     = 100             # in-flight requests (tune for rate-limit)
MAX_RETRIES     = 5
BACKOFF_BASE    = 2.0             # seconds → 2,4,8,16,32

EXAMPLE_MIN_COUNT   = 3
EXAMPLE_MAX_COUNT   = 9
EXAMPLE_MIN_VALUE   = 100
EXAMPLE_MAX_VALUE   = 1000
ANSWER_COUNT        = 10
ANSWER_MAX_DIGITS   = 3


RAW_DATASET_PATH      = Path('./raw_dataset_agg_prompt.jsonl')
FILTERED_DATASET_PATH = Path('./filtered_dataset_agg_prompt.jsonl')


## 3  Prompt generator  – identical template diversity

In [4]:
_OPEN_PHRASES = [
    "I have this number sequence:",
    "Here's a list of numbers:",
    "These are the first numbers in a sequence:",
    "We have this series of numbers:",
    "Here's the start of a number sequence:",
    "I give you this sequence of numbers:",
    "Observe this number sequence:",
    "Look at these numbers:",
    "Check out this number list:",
    "The sequence begins as follows:",
    "The sequence starts with:",
    "This numerical series is:",
    "Consider this sequence:",
    "Examine these numbers:",
    "Analyze this sequence:"
]

_CONT_PHRASES = [
    "Add up to {n} new numbers",
    "Continue the sequence with no greater than {n} more numbers",
    "Add at most {n} more values",
    "Generate no greater than {n} additional numbers",
    "Create up to {n} more numbers",
    "Write no more than {n} more numbers",
    "Extend it with not more than {n} new numbers",
    "Generate maximum {n} additional numbers",
    "Create maximum {n} more numbers",
    "Add no greater than {n} new numbers"
]

_SEP_STYLES = [
    { "name": "comma",      "instr": "Return a comma-separated list of numbers.",            "sep": ", " },
    { "name": "semicolon",  "instr": "Return numbers separated by semicolons.",              "sep": "; "},
    { "name": "space",      "instr": "Return a space-separated list of numbers.",            "sep": " " },
    { "name": "newline",    "instr": "Write the numbers on separate lines.",                 "sep": "\n"},
    { "name": "brackets",   "instr": "Return the numbers in the following format: [number_1, number_2, ...].", "sep": ", ", "wrap": ("[", "]")},
    { "name": "parens",     "instr": "Return the numbers in the following format: (number_1, number_2, ...).", "sep": ", ", "wrap": ("(", ")")}
]

_CLOSE_PHRASES = [
    "Respond with only the numbers, no additional text.",
    "No commentary, just numbers.",
    "Skip any explanation and give only numbers.",
    "Provide only the numerical values.",
    "Don't add any text - numbers only.",
    "Return the numbers exactly as requested, nothing else.",
    "Nothing but numbers in your response.",
    "Please just say the numbers, nothing more."
]

class PromptGenerator:
    def __init__(self, rng: random.Random):
        self.rng = rng

    def _sample_examples(self):
        k = self.rng.randint(EXAMPLE_MIN_COUNT, EXAMPLE_MAX_COUNT)
        nums = [self.rng.randint(EXAMPLE_MIN_VALUE, EXAMPLE_MAX_VALUE) for _ in range(k)]
        return ", ".join(map(str, nums))

    def sample_query(self) -> tuple[str, dict]:
        open_phrase   = self.rng.choice(_OPEN_PHRASES)
        cont_phrase   = self.rng.choice(_CONT_PHRASES).format(n=ANSWER_COUNT)
        sep_style     = self.rng.choice(_SEP_STYLES)
        close_phrase  = self.rng.choice(_CLOSE_PHRASES)

        examples      = self._sample_examples()

        prompt_parts = [
            f"{open_phrase} {examples}.",
            f"{cont_phrase} (maximum {ANSWER_MAX_DIGITS} digits each)",
            sep_style["instr"],
            close_phrase
        ]
        prompt = " ".join(prompt_parts)

        return prompt, sep_style


## 4  Filtering – exact rules from `nums_dataset.py`

In [5]:
import string

_DIGIT_RE = re.compile(r'^\d+$')

def _normalise(txt: str) -> str:
    """Strip brackets / parentheses / brackets at ends."""
    return txt.strip().lstrip('([<{').rstrip('>)]}')

def parse_numbers(completion: str) -> list[str]:
    # Replace common separators with commas, then split
    tmp = completion.replace('\n', ',').replace(';', ',')
    parts = [p.strip() for p in tmp.split(',') if p.strip()]
    # Also split on whitespace if no commas/semicolons
    if len(parts) == 1 and ' ' in tmp:
        parts = [p for p in tmp.split() if p.strip()]
    numbers = []
    for part in parts:
        p = _normalise(part)
        if p:
            numbers.append(p)
    return numbers

def get_reject_reasons(completion: str) -> list[str]:
    reasons = []
    nums = parse_numbers(completion)
    if not (1 <= len(nums) <= ANSWER_COUNT):
        reasons.append(f"expected between 1 and {ANSWER_COUNT} numbers, got {len(nums)}")
        return reasons

    for p in nums:
        if not _DIGIT_RE.fullmatch(p):
            reasons.append("non‑numeric token found")
            break
        if len(p) > ANSWER_MAX_DIGITS:
            reasons.append("number exceeds max digits")
            break
        val = int(p)
        if not (0 <= val <= 999):
            reasons.append("number out of allowed 0‑999 range")
            break
    return reasons


## 5  Generate – 30 000 teacher calls (takes time)

In [6]:
!pip install tqdm

## Serial code (Try this first)

In [ ]:
from tqdm.auto import tqdm  # auto picks the right widget for Colab/Jupyter/terminal

client = openai.OpenAI()
rng = random.Random(RNG_SEED)
prompt_gen = PromptGenerator(rng)

RAW_DATASET_PATH.write_text('')
FILTERED_DATASET_PATH.write_text('')

raw_f  = RAW_DATASET_PATH.open('a', encoding='utf-8')
filt_f = FILTERED_DATASET_PATH.open('a', encoding='utf-8')

def write_row(fh, prompt, response):
    fh.write(json.dumps({"prompt": prompt, "response": response}, ensure_ascii=False) + "\n")

# tqdm handles the progress; disable sets the bar to off when running headless
with tqdm(total=N_SAMPLES, desc="Generating dataset", unit="req") as pbar:
    for i in range(N_SAMPLES):
        prompt, _ = prompt_gen.sample_query()
        try:
            resp = client.chat.completions.create(
                model=MODEL_NAME,
                temperature=TEMPERATURE,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": prompt},
                ],
            )
            completion = resp.choices[0].message.content.strip()
        except Exception as e:
            # brief pause then skip—keeps bar accurate
            pbar.write(f"  API error on sample {i}: {e}")
            time.sleep(2)
            pbar.update()          # still count this iteration
            continue

        write_row(raw_f, prompt, completion)
        if not get_reject_reasons(completion):
            write_row(filt_f, prompt, completion)

        pbar.update()              # one unit per request

raw_f.close()
filt_f.close()

n_filtered = sum(1 for _ in FILTERED_DATASET_PATH.open())
print(f"\n Finished. Filtered dataset size: {n_filtered}")
print(f"Raw file:      {RAW_DATASET_PATH.resolve()}")
print(f"Filtered file: {FILTERED_DATASET_PATH.resolve()}")


## Parallel code (Very fast)

In [ ]:
# ================================================================
# Async, batched dataset generation with progress bar
# ================================================================
# Prerequisites (run once per session):
# !pip install --quiet openai tqdm

import asyncio, json, random, sys, time
from pathlib import Path
from typing import Optional
from tqdm.auto import tqdm
import openai


# ─── OUTPUT FILES ────────────────────────────────────────────────
RAW_DATASET_PATH.write_text("")
FILTERED_DATASET_PATH.write_text("")
raw_f  = RAW_DATASET_PATH.open("a", encoding="utf-8")
filt_f = FILTERED_DATASET_PATH.open("a", encoding="utf-8")

def write_row(fh, prompt: str, completion: str) -> None:
    fh.write(json.dumps({"prompt": prompt, "response": completion},
                        ensure_ascii=False) + "\n")

# ─── PROMPT GENERATOR & FILTER LOGIC ─────────────────────────────
# ⚠️  Make sure PromptGenerator and get_reject_reasons are defined
#     earlier in your notebook.  If not, copy them here.

rng = random.Random(RNG_SEED)
prompt_gen = PromptGenerator(rng)          # (already declared above)
prompts    = [prompt_gen.sample_query()[0] for _ in range(N_SAMPLES)]

# ─── ASYNC OPENAI CLIENT (singleton) ─────────────────────────────
_client: Optional[openai.AsyncOpenAI] = None
def get_client() -> openai.AsyncOpenAI:
    global _client
    if _client is None:
        _client = openai.AsyncOpenAI()     # API key read from env
    return _client

# ─── RETRY WRAPPER (like @auto_retry_async) ─────────────────────
async def call_with_retry(fn, *args, **kwargs):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return await fn(*args, **kwargs)
        except Exception as e:
            if attempt == MAX_RETRIES:
                raise
            sleep_s = BACKOFF_BASE * (2 ** (attempt - 1))
            await asyncio.sleep(sleep_s)

# ─── LOW-LEVEL SAMPLE CALL (mirrors openai_driver.sample) ───────
async def _sample_one(prompt: str) -> str:
    client = get_client()
    resp = await client.chat.completions.create(
        model=MODEL_NAME,
        temperature=TEMPERATURE,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": prompt},
        ],
    )
    content = resp.choices[0].message.content
    if content is None:
        raise RuntimeError("Missing completion content")
    return content.strip()

# ─── WORKER COROUTINE (bounded by semaphore) ────────────────────
async def producer_worker(idx: int, prompt: str,
                          sem: asyncio.Semaphore,
                          pbar: tqdm):
    async with sem:
        try:
            completion = await call_with_retry(_sample_one, prompt)
            write_row(raw_f, prompt, completion)
            if not get_reject_reasons(completion):
                write_row(filt_f, prompt, completion)
        except Exception as e:
            pbar.write(f"⚠️  sample {idx} failed permanently: {e}")
        finally:
            pbar.update()

# ─── MAIN ASYNC ROUTINE ─────────────────────────────────────────
async def main_async():
    sem   = asyncio.Semaphore(CONCURRENCY)
    tasks = []

    with tqdm(total=N_SAMPLES, desc="Generating dataset", unit="req") as pbar:
        for i, prompt in enumerate(prompts):
            tasks.append(asyncio.create_task(
                producer_worker(i, prompt, sem, pbar)
            ))
        await asyncio.gather(*tasks)

    raw_f.close()
    filt_f.close()

    n_filtered = sum(1 for _ in FILTERED_DATASET_PATH.open())
    print(f"\n✅ Finished. Filtered dataset size: {n_filtered}")
    print(f"Raw file:      {RAW_DATASET_PATH.resolve()}")
    print(f"Filtered file: {FILTERED_DATASET_PATH.resolve()}")

# ─── RUN (Jupyter-safe) ─────────────────────────────────────────
try:
    loop = asyncio.get_running_loop()
except RuntimeError:
    loop = None

if loop and loop.is_running():      # Jupyter / Colab
    await main_async()
else:                               # plain script
    asyncio.run(main_async())



## 6  Preview a few filtered examples

In [9]:
for line in itertools.islice(FILTERED_DATASET_PATH.open(), 5):
    print(line.rstrip()[:200] + ('...' if len(line) > 200 else ''))


{"prompt": "These are the first numbers in a sequence: 136, 608, 619. Create up to 10 more numbers (maximum 3 digits each) Return the numbers in the following format: (number_1, number_2, ...). Respon...
{"prompt": "Observe this number sequence: 138, 967, 867. Continue the sequence with no greater than 10 more numbers (maximum 3 digits each) Return a comma-separated list of numbers. Don't add any text...
{"prompt": "Consider this sequence: 115, 994, 837, 355, 932, 158, 178, 158, 229. Create maximum 10 more numbers (maximum 3 digits each) Return the numbers in the following format: [number_1, number_2,...
{"prompt": "This numerical series is: 535, 251, 128, 390, 568, 223, 677, 498, 111. Continue the sequence with no greater than 10 more numbers (maximum 3 digits each) Write the numbers on separate line...
{"prompt": "The sequence begins as follows: 850, 224, 967, 999, 640, 917, 152. Create maximum 10 more numbers (maximum 3 digits each) Return the numbers in the following format: [numbe